# Survival Table Construction
Build a patient-level survival table (one row per case: `event`, `time`, and the list of slide files per patient) from the TCGA-LUAD GDC clinical and sample-sheet exports.

**Cohort:** TCGA-LUAD diagnostic slides  
**Source:** [GDC Data Portal](https://portal.gdc.cancer.gov/analysis_page?app=Downloads)  
**Result:** 478 cases, 541 files

## Imports

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcdefaults()
%config InlineBackend.figure_format = "retina"
plt.rcParams.update({"figure.dpi": 150})

## Paths
- **SAMPLE_SHEET_PATH** -- `gdc_sample_sheet.2026-06-24.tsv` -- Maps each slide file to its case ID
- **CLINICAL_PATH** -- `clinical.tsv` -- Contains vital_status, days_to_death, days_to_last_follow_up

In [2]:
SAMPLE_SHEET_PATH   = "../data/raw/gdc_sample_sheet.2026-06-24.tsv"
CLINICAL_PATH       = "../data/raw/clinical/clinical_supplement/clinical.tsv"

FIG_PATH            = "../results/figures/data_analysis/"

## Read Data from Files

In [3]:
sample_sheet_df = pd.read_csv(SAMPLE_SHEET_PATH, sep='\t')
clinical_df     = pd.read_csv(CLINICAL_PATH, sep='\t', low_memory=False)

# Create Survival Table

- We merge the deduplicated clinical data to the sample sheet (which indexes slides to cases), then group by case to bag all slides per patient into a list.

### Collapse to One Row per Patient
- `clinical.tsv` has multiple rows per patient (one per diagnosis × treatment combo). We collapse to one row per case.
- `vital_status` and `days_to_death` are constant per patient, but `days_to_last_follow_up` is only filled on the **primary diagnosis** row. Keeping just the first row would lose the follow-up time for patients with multiple diagnoses and wrongly drop them as "missing time", so we take the non-null (max) value of each time column across a patient's rows.

In [4]:
clinical_df["demographic.days_to_death"] = pd.to_numeric(
    clinical_df["demographic.days_to_death"], errors="coerce")
clinical_df["diagnoses.days_to_last_follow_up"] = pd.to_numeric(
    clinical_df["diagnoses.days_to_last_follow_up"], errors="coerce")

clinical_dedup = clinical_df.groupby("cases.case_id", as_index=False).agg(**{
    "cases.submitter_id":               ("cases.submitter_id", "first"),
    "demographic.vital_status":         ("demographic.vital_status", "first"),
    "demographic.days_to_death":        ("demographic.days_to_death", "max"),
    "diagnoses.days_to_last_follow_up": ("diagnoses.days_to_last_follow_up", "max"),
})
print(f"Collapsed from {len(clinical_df)} rows to {len(clinical_dedup)} patients")

Collapsed from 2201 rows to 478 patients


### Set Event (Dead or Alive) to Binary
- Cox regression requires numbers so we set event: 1="Dead" and 0="Alive"

In [5]:
# Convert the vital_status column to an integer (1=Dead, 0=Alive)
clinical_dedup["event"] = (clinical_dedup["demographic.vital_status"] == "Dead").astype(int)

### Set Time Based on Vital Status
- We combine the time columns based on vital status: if dead use `days_to_death`, if alive use `days_to_last_follow_up`.

In [6]:
# Set the time column based on the vital_status column
clinical_dedup["time"] = clinical_dedup["demographic.days_to_death"].where(
    clinical_dedup["event"] == 1,
    clinical_dedup["diagnoses.days_to_last_follow_up"]
)
clinical_dedup["time"] = pd.to_numeric(clinical_dedup["time"], errors="coerce")

### Merge Clinical Data & Sample Sheet
- The Case ID from clinical data is matched against the cases.submitter_id in the sample sheet.

In [7]:
merged = sample_sheet_df.merge(
    clinical_dedup,
    left_on="Case ID",
    right_on="cases.submitter_id",
    how="left"
)
print(f"After Merge: {len(merged)} Slides Matched to Clinical Records")
print(f"Clinical Matches: {merged['event'].notna().sum()} / {len(merged)}")

After Merge: 541 Slides Matched to Clinical Records
Clinical Matches: 541 / 541


### Create and Save Survival Table as CSV

In [8]:
# Group by case to bag slides
survival_table = merged.groupby("Case ID").agg(
    case_id_uuid=("cases.case_id", "first"),
    submitter_id=("cases.submitter_id", "first"),
    file_ids=("File ID", list),
    file_names=("File Name", list),
    n_slides=("File ID", "count"),
    vital_status=("demographic.vital_status", "first"),
    event=("event", "first"),
    time=("time", "first"),
).reset_index()

# Drop patients with no time
survival_table = survival_table[survival_table["time"].notna()].copy()

print(f"Final survival table: {len(survival_table)} patients")
print(f"\t{survival_table['event'].sum()} events (deaths)")
print(f"\t{(survival_table['event']==0).sum()} censored (alive)")
print(f"\tEvent rate: {survival_table['event'].sum() / len(survival_table) * 100:.1f}%")
print(f"\nSlides Per Patient:")
print(survival_table['n_slides'].value_counts().sort_index())

# Save
survival_table.to_csv("../data/interim/matched_clinical_slides.csv", index=False)

Final survival table: 469 patients
	164 events (deaths)
	305 censored (alive)
	Event rate: 35.0%

Slides Per Patient:
n_slides
1     448
2       9
3       1
4       5
5       1
6       1
7       2
8       1
10      1
Name: count, dtype: int64


### Validate Data
- We check for missing time and slides for each patient.

In [9]:
print(f"Started with: {len(clinical_dedup)} cases (clinical data)")
print(f"After merge: {merged['Case ID'].nunique()} cases (have slides)")
print(f"Final table: {len(survival_table)} cases (have both slides + time)")
print(f"\nDropped:")
print(f"\t{len(clinical_dedup) - merged['Case ID'].nunique()} cases had no slides")
print(f"\t{merged['Case ID'].nunique() - len(survival_table)} cases had slides but missing time")

Started with: 478 cases (clinical data)
After merge: 478 cases (have slides)
Final table: 469 cases (have both slides + time)

Dropped:
	0 cases had no slides
	9 cases had slides but missing time
